# Online Retail Sales – Data Cleaning

## 1. Load and inspect the data

In [47]:
import pandas as pd
df = pd.read_csv("online_retail_messy.csv")
print(df.shape)
df.info()
print(df.head(10))
print(df.isna().sum())
print(df.duplicated().sum())

(6885, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6885 entries, 0 to 6884
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Order ID          6873 non-null   object
 1   customer_id       6736 non-null   object
 2   Customer Name     6581 non-null   object
 3   City              6496 non-null   object
 4   state             6654 non-null   object
 5   Order Date        6821 non-null   object
 6   product_category  6873 non-null   object
 7   Product Name      6873 non-null   object
 8   Quantity          6811 non-null   object
 9   unit_price        6750 non-null   object
 10  discount          6873 non-null   object
 11  payment_method    6692 non-null   object
 12  Order_Status      6873 non-null   object
 13  delivery days     6406 non-null   object
 14  Rating            5807 non-null   object
 15  phone             6205 non-null   object
 16  email             6660 non-null   object
dtypes: 

## 2. Clean column names


In [48]:
df.columns = (df.columns
              .str.strip()
              .str.lower()
              .str.replace(" ", "_"))
print(df.columns.tolist())

['order_id', 'customer_id', 'customer_name', 'city', 'state', 'order_date', 'product_category', 'product_name', 'quantity', 'unit_price', 'discount', 'payment_method', 'order_status', 'delivery_days', 'rating', 'phone', 'email']


## 3. Check spelling variations

In [49]:
text_cols = ["city", "state", "payment_method", "order_status", "product_category", "customer_name", "product_name"]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.title()
print(df["order_status"].value_counts(dropna=False))

order_status
Delivered    5476
Returned      703
Cancelled     369
Delivrd       190
Delivere       50
Return         27
Returnd        19
Canceled       14
<NA>           12
Returne        11
Cancel         10
Cancelle        4
Name: count, dtype: Int64


## 4. Standardize text columns

In [50]:
text_cols = ["city", "state", "payment_method", "order_status", "product_category", "customer_name", "product_name"]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.title()
print(df["order_status"].value_counts(dropna=False))

order_status
Delivered    5476
Returned      703
Cancelled     369
Delivrd       190
Delivere       50
Return         27
Returnd        19
Canceled       14
<NA>           12
Returne        11
Cancel         10
Cancelle        4
Name: count, dtype: Int64


## 5. Fix spelling mistakes

In [51]:
status_map = {"Delivrd": "Delivered", "Delivere": "Delivered",
              "Return": "Returned", "Returnd": "Returned", "Returne": "Returned",
              "Canceled": "Cancelled", "Cancel": "Cancelled", "Cancelle": "Cancelled"}
df["order_status"] = df["order_status"].replace(status_map)

cat_map = {"Electronic": "Electronics", "Fashon": "Fashion", "Fasion": "Fashion", "Fashio": "Fashion",
           "Beuty": "Beauty", "Beaut": "Beauty", "Sport": "Sports",
           "Home&Kitchen": "Home & Kitchen", "Home And Kitchen": "Home & Kitchen",
           "Home & Kitche": "Home & Kitchen"}
df["product_category"] = df["product_category"].replace(cat_map)

print(df["order_status"].value_counts(dropna=False))
print(df["product_category"].value_counts(dropna=False))

order_status
Delivered    5716
Returned      760
Cancelled     397
<NA>           12
Name: count, dtype: Int64
product_category
Electronics       2106
Fashion           1681
Home & Kitchen    1337
Beauty             936
Sports             813
<NA>                12
Name: count, dtype: Int64


## 6. Fix payment_method and state

In [52]:
pay_map = {"Cod": "Cash On Delivery"}
df["payment_method"] = df["payment_method"].replace(pay_map)

state_map = {"Mh": "Maharashtra", "Maharastra": "Maharashtra",
             "Ka": "Karnataka", "Karnatak": "Karnataka",
             "Dl": "Delhi", "New Delhi": "Delhi",
             "Tn": "Tamil Nadu", "Tamilnadu": "Tamil Nadu"}
df["state"] = df["state"].replace(state_map)

print(df["payment_method"].value_counts(dropna=False))
print(df["state"].value_counts(dropna=False))

payment_method
Upi                 2468
Cash On Delivery    1317
Credit Card         1225
Debit Card           909
Net Banking          466
Wallet               307
<NA>                 193
Name: count, dtype: Int64
state
Maharashtra    1879
Karnataka      1514
Delhi           847
Telangana       641
Tamil Nadu      567
Rajasthan       363
West Bengal     311
Kerala          269
Gujarat         263
<NA>            231
Name: count, dtype: Int64


## 7. Fix city
Fixed truncated names (Bengalur, Mumba). Replaced placeholders (Unknown, -, N/A) with missing values.

In [53]:
city_map = {"Bengalur": "Bengaluru", "Mumba": "Mumbai", "Hyderaba": "Hyderabad",
            "Jaipu": "Jaipur", "Chenna": "Chennai", "Kolkat": "Kolkata",
            "Delh": "Delhi", "Koch": "Kochi", "Ahmedaba": "Ahmedabad"}
df["city"] = df["city"].replace(city_map)

for col in ["city", "state", "payment_method", "order_status", "product_category"]:
    df[col] = df[col].replace(["Unknown", "-", "N/A", "Na", ""], pd.NA)

print(df["city"].value_counts(dropna=False))

city
Bengaluru    1469
Mumbai       1225
Delhi         809
Hyderabad     629
Pune          612
Chennai       550
<NA>          414
Jaipur        354
Kolkata       302
Kochi         263
Ahmedabad     258
Name: count, dtype: Int64


## 8. Remove duplicate orders
Same order appeared more than once. Kept the first row for each order_id.

In [54]:
print("Rows before:", len(df))
print("Duplicate order_id:", df["order_id"].duplicated().sum())

df["order_id"] = df["order_id"].astype("string").str.strip().str.upper()
df = df.drop_duplicates(subset="order_id", keep="first")

print("Rows after:", len(df))
print("Missing order_id:", df["order_id"].isna().sum())

Rows before: 6885
Duplicate order_id: 584
Rows after: 6216
Missing order_id: 1


## 9. Fix order_date
Dates came in many formats. Converted all to a single date type. Future dates (e.g. 2030) are impossible, so set to missing.

In [55]:
import re
df["order_date"] = df["order_date"].astype("string").str.strip()

def parse_date(x):
    if pd.isna(x):
        return pd.NaT
    if re.fullmatch(r"\d{5}", x):   # Excel serial number, e.g. 45873
        return pd.Timestamp("1899-12-30") + pd.Timedelta(days=int(x))
    return pd.to_datetime(x, dayfirst=True, errors="coerce", format="mixed")

df["order_date"] = df["order_date"].apply(parse_date)

df.loc[df["order_date"] > "2025-12-31", "order_date"] = pd.NaT   # future dates are impossible
print(df["order_date"].isna().sum(), "missing dates")
print("Earliest:", df["order_date"].min())
print("Latest:", df["order_date"].max())

54 missing dates
Earliest: 2024-01-01 00:00:00
Latest: 2025-12-31 00:00:00


## 10. Convert text-like numbers to real numbers
quantity, unit_price, discount, delivery_days and rating contained text (words, symbols, units). Extracted the numbers.

In [56]:
words = {"one": "1", "two": "2", "three": "3", "four": "4", "five": "5"}

# quantity: words -> numbers
q = df["quantity"].astype("string").str.strip().str.lower().replace(words)
df["quantity"] = pd.to_numeric(q.str.extract(r"(-?\d+\.?\d*)")[0], errors="coerce")

# unit_price: remove Rs., INR, ₹ and fix comma decimals
p = (df["unit_price"].astype("string").str.strip()
     .str.replace(",", ".", regex=False)
     .str.extract(r"(-?\d+\.?\d*)")[0])
df["unit_price"] = pd.to_numeric(p, errors="coerce")

# discount: remove % / percent; 0.10 -> 10
d = df["discount"].astype("string").str.extract(r"(\d+\.?\d*)")[0]
d = pd.to_numeric(d, errors="coerce")
df["discount"] = d.where(~((d > 0) & (d <= 1)), d * 100)

# delivery_days: "3d", "4 days" -> 3, 4 ; "pending" -> missing
df["delivery_days"] = pd.to_numeric(
    df["delivery_days"].astype("string").str.extract(r"(-?\d+\.?\d*)")[0], errors="coerce")

# rating: words and "4/5", "4 stars" -> 4
r = df["rating"].astype("string").str.strip().str.lower().replace(words)
df["rating"] = pd.to_numeric(r.str.extract(r"(\d+\.?\d*)")[0], errors="coerce")

print(df[["quantity", "unit_price", "discount", "delivery_days", "rating"]].dtypes)
print(df[["quantity", "unit_price", "discount", "delivery_days", "rating"]].describe())

quantity         Float64
unit_price       Float64
discount         Float64
delivery_days    Float64
rating           Float64
dtype: object
       quantity    unit_price  discount  delivery_days    rating
count    6160.0        6101.0    6215.0         5667.0    5230.0
mean   1.628896   4606.677255  5.061947       3.630669  3.821797
std    3.056051  22235.300087  7.594498       3.392544  0.987386
min        -2.0       -1500.0       0.0           -3.0       0.0
25%         1.0       2298.44       0.0            2.0       3.0
50%         1.0       3777.82       0.0            3.0       4.0
75%         2.0       5641.29      10.0            5.0       5.0
max       100.0      999999.0      30.0          120.0      10.0


## 11. Remove test rows and fix impossible values
Removed test entries (order_id starting with TEST). Set impossible values to missing: quantity outside 1-10, price <= 0 or > 100000, delivery_days outside 0-30, rating outside 1-5.

In [57]:
# 1) remove test rows
before = len(df)
df = df[~df["order_id"].astype("string").str.upper().str.startswith("TEST", na=False)]
print("Test rows removed:", before - len(df))

# 2) impossible values -> missing
df.loc[(df["quantity"] < 1) | (df["quantity"] > 10), "quantity"] = pd.NA
df.loc[(df["unit_price"] <= 0) | (df["unit_price"] > 100000), "unit_price"] = pd.NA
df.loc[(df["delivery_days"] < 0) | (df["delivery_days"] > 30), "delivery_days"] = pd.NA
df.loc[(df["rating"] < 1) | (df["rating"] > 5), "rating"] = pd.NA

print(df[["quantity", "unit_price", "delivery_days", "rating"]].describe())

Test rows removed: 15
       quantity   unit_price  delivery_days    rating
count    6133.0       6077.0         5644.0    5210.0
mean   1.536279  4132.439185       3.530829  3.817658
std    0.898391  2616.036303       1.694224  0.978437
min         1.0       630.09            1.0       1.0
25%         1.0      2325.46            2.0       3.0
50%         1.0      3784.18            3.0       4.0
75%         2.0      5642.44            5.0       5.0
max         5.0     99999.99           11.0       5.0


## 12. Standardize customer_id and product_name
customer_id had formats like C1709, c1709, C-1709. Made all C#### format. Fixed product name variants.

In [58]:
# customer_id: C1709 / c1709 / C-1709 -> C1709
df["customer_id"] = (df["customer_id"].astype("string").str.upper()
                     .str.replace(r"[^A-Z0-9]", "", regex=True))

# product_name variants
prod_map = {"Running Shoe": "Running Shoes", "Airfryer": "Air Fryer",
            "Smartwatch": "Smart Watch", "Faceserum": "Face Serum",
            "Wireless Earbud": "Wireless Earbuds"}
df["product_name"] = df["product_name"].replace(prod_map)

print("Unique customer_id:", df["customer_id"].nunique())
print("Unique products:", df["product_name"].nunique())

Unique customer_id: 804
Unique products: 24


## 13. Check missing values

In [59]:
print(df.isna().sum())
print((df.isna().mean() * 100).round(1))

order_id              1
customer_id         124
customer_name       260
city                356
state               197
order_date           54
product_category      1
product_name          1
quantity             68
unit_price          124
discount              1
payment_method      166
order_status          1
delivery_days       557
rating              991
phone               617
email               196
dtype: int64
order_id             0.0
customer_id          2.0
customer_name        4.2
city                 5.7
state                3.2
order_date           0.9
product_category     0.0
product_name         0.0
quantity             1.1
unit_price           2.0
discount             0.0
payment_method       2.7
order_status         0.0
delivery_days        9.0
rating              16.0
phone               10.0
email                3.2
dtype: float64


## 14. Create revenue, drop personal columns, save cleaned file
revenue = quantity x unit_price x (1 - discount/100). Dropped phone and email (not needed for analysis).

In [60]:
# make sure numeric columns are real numbers
for col in ["quantity", "unit_price", "discount", "delivery_days", "rating"]:
    df[col] = df[col].astype(float)

# revenue after discount (missing if quantity or price is missing)
df["discount"] = df["discount"].fillna(0)
df["revenue"] = (df["quantity"] * df["unit_price"] * (1 - df["discount"] / 100)).round(2)

# drop personal columns
df = df.drop(columns=["phone", "email"])

# final checks
print(df.shape)
print(df.isna().sum())
print(df.head())

# save
df.to_csv("online_retail_cleaned.csv", index=False)

(6201, 16)
order_id              1
customer_id         124
customer_name       260
city                356
state               197
order_date           54
product_category      1
product_name          1
quantity             68
unit_price          124
discount              0
payment_method      166
order_status          1
delivery_days       557
rating              991
revenue             189
dtype: int64
    order_id customer_id    customer_name       city        state order_date  \
0  ORD201736       C1310     Pavan Poojar  Bengaluru    Karnataka 2025-11-25   
1  ORD201610       C1407  Pranali Gaikwad     Mumbai  Maharashtra 2025-11-06   
2  ORD201897       C1029      Mayur Pawar       <NA>  Maharashtra 2024-05-26   
3  ORD201249       C1627   Midhun Nambiar      Kochi       Kerala 2025-05-31   
4  ORD206175       C1054       Veena Naik  Bengaluru    Karnataka 2025-12-17   

  product_category      product_name  quantity  unit_price  discount  \
0           Beauty        Makeup Kit   

In [61]:
from google.colab import files
files.download("online_retail_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>